### Robust Crawling

**Goal:** Build a "Spider" that can crawl through hundreds of pages and handle messy data without crashing.

### Pagination (The Loop)
Most websites split data across multiple pages. We need a loop that updates the URL automatically. We will use **Page 3** of the bookstore as our testing ground to see how to move forward to Page 4 and beyond.

### Method 1: The Iterative URL (The "For" Loop)
**Best for:** Sites where the page number is clearly in the URL.
**Logic:** We notice the pattern `page-3.html`, `page-4.html`. We can generate these mathematically.

In [1]:
import requests
from bs4 import BeautifulSoup
import time

# 1. Inspect the pattern
# Target: http://books.toscrape.com/catalogue/page-3.html

# We replace the number '3' with the placeholder '{}'
base_url = "http://books.toscrape.com/catalogue/page-{}.html"

# Let's start scraping from Page 3 and go to Page 5
for page_num in range(3, 6): 
    
    # Inject the number into the URL
    current_url = base_url.format(page_num)
    
    print(f"Scraping: {current_url}")
    
    response = requests.get(current_url)
    
    # (Insert Module 2 & 3 extraction logic here...)
    
    # CRITICAL: Pause between requests to be polite
    time.sleep(2) 

Scraping: http://books.toscrape.com/catalogue/page-3.html
Scraping: http://books.toscrape.com/catalogue/page-4.html
Scraping: http://books.toscrape.com/catalogue/page-5.html


### Method 2: The "Next" Button (The "While" Loop)
**Best for:** When you don't know the last page number, or you are starting in the middle (like Page 3) and just want to keep going until the end.
**Logic:** We scrape Page 3, find the "next" button, extract the link to Page 4, and update our target.

In [2]:
import requests
from bs4 import BeautifulSoup
import time

# Start explicitly at Page 3
url = "http://books.toscrape.com/catalogue/page-3.html"

while True: # Run forever until we 'break'
    print(f"Scraping: {url}")
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    
    # 1. Extract Data from current page
    books = soup.select(".product_pod")
    # ... extraction logic ...

    # 2. Look for the "Next" button
    # On Page 3, the HTML looks like: <li class="next"><a href="page-4.html">next</a></li>
    next_button = soup.select_one("li.next a")

    if next_button:
        # 3. Extract the href
        # Result: "page-4.html"
        next_page_url = next_button["href"]
        
        # 4. Construct the full URL
        # Since we are in the 'catalogue' directory, we append the new page to the base path.
        url = "http://books.toscrape.com/catalogue/" + next_page_url
        
        time.sleep(2)
    else:
        # 5. No button found? We are at the end.
        print("No more pages found. Exiting.")
        break

Scraping: http://books.toscrape.com/catalogue/page-3.html
Scraping: http://books.toscrape.com/catalogue/page-4.html
Scraping: http://books.toscrape.com/catalogue/page-5.html
Scraping: http://books.toscrape.com/catalogue/page-6.html
Scraping: http://books.toscrape.com/catalogue/page-7.html
Scraping: http://books.toscrape.com/catalogue/page-8.html
Scraping: http://books.toscrape.com/catalogue/page-9.html
Scraping: http://books.toscrape.com/catalogue/page-10.html
Scraping: http://books.toscrape.com/catalogue/page-11.html
Scraping: http://books.toscrape.com/catalogue/page-12.html
Scraping: http://books.toscrape.com/catalogue/page-13.html
Scraping: http://books.toscrape.com/catalogue/page-14.html
Scraping: http://books.toscrape.com/catalogue/page-15.html
Scraping: http://books.toscrape.com/catalogue/page-16.html
Scraping: http://books.toscrape.com/catalogue/page-17.html
Scraping: http://books.toscrape.com/catalogue/page-18.html
Scraping: http://books.toscrape.com/catalogue/page-19.html
Scra

## Handling Missing Data
In the real world, data is messy. On `books.toscrape.com`, some books have long titles that get cut off, and in a real scenario, a price or rating might be missing.

If you try to access `.text` on a tag that doesn't exist, Python throws an `AttributeError` and your script **crashes**.

### The Problem: The "NoneType" Crash

In [3]:
# Scenario: Using select_one()
tag = soup.select_one(".price") 

# If the tag exists, 'tag' is a Soup Object.
# If the tag is missing, 'tag' is None.

print(tag.text) 
# CRASH: "AttributeError: 'NoneType' object has no attribute 'text'"

AttributeError: 'NoneType' object has no attribute 'text'

### The Solution: `try` / `except` Blocks
We wrap the extraction code in a `try` block. If it fails, Python jumps to the `except` block, assigns a default value ("N/A"), and keeps running.

In [4]:
# Dummy HTML simulating a broken book card on Page 3
html = """
<div class="product">
    <h2>The Book of Python</h2>
    <!-- Price is missing here! -->
</div>
"""
soup = BeautifulSoup(html, "html.parser")
card = soup.select_one(".product")

item = {}

# 1. Title (Usually safe, but good to be careful)
try:
    item['title'] = card.select_one("h2").get_text(strip=True)
except AttributeError:
    item['title'] = "Unknown Title"

# 2. Price (Risky - The price tag is missing in our HTML above)
try:
    # This line will fail because .select_one(".price") returns None
    item['price'] = card.select_one(".price").get_text(strip=True)
except AttributeError:
    # Python catches the error here instead of crashing
    item['price'] = "N/A" 
    print("Warning: Price missing for this item")

print(item)
# Output: {'title': 'The Book of Python', 'price': 'N/A'}

{'title': 'The Book of Python', 'price': 'N/A'}


### Alternative: The "If" Check
For simple checks, you can verify if the element exists before asking for text.

In [5]:
price_tag = card.select_one(".price")

if price_tag:
    price = price_tag.get_text(strip=True)
else:
    price = "0.00"


## Network Reliability (Status Codes)
Sometimes the error isn't in your code; it's in the server (e.g., the website goes down, or you get banned).

We should always check `response.status_code` before feeding the content to BeautifulSoup.

In [6]:
response = requests.get(url)

# Raise an error immediately if the status is 4xx or 5xx
try:
    response.raise_for_status() 
    # If we get here, the request was successful (200)
    soup = BeautifulSoup(response.content, "html.parser")
except requests.exceptions.HTTPError as e:
    print(f"Skipping page due to network error: {e}")